In [0]:
%sql
SHOW SCHEMAS IN covid19

In [0]:
%sql
SHOW TABLES IN covid19.silver

In [0]:
%sql
-- Ver una muestra de los datos raw
SELECT *
FROM covid19.bronze.who_covid_19_global_daily_data
LIMIT 10

In [0]:
#Estamos viendo los datos como estan antes de limpiarlos
#Esto para la tabla del 2020
df_bronze = spark.table("covid19.bronze.mortalidad_categorias_costa_rica_2020")
df_bronze.printSchema()
display(df_bronze.limit(10))

In [0]:
%sql
SELECT * FROM covid19.bronze.who_mortality LIMIT 3

In [0]:
import requests

url = "https://1drv.ms/x/c/9b3df75ac5d97177/IQDZZUKSZbzIR5igaZ3f9CzHAbVWmwADCqD4wWUP4-l-bR0?e=WWwKGa"
download_url = url.split("?")[0] + "?download=1"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
response = requests.get(download_url, headers=headers)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Primeros 500 caracteres del contenido:")
print(response.content[:500])

In [0]:
import pandas as pd
import io

clean_text = response.content.decode('utf-8-sig', errors='ignore')

# Vamos a ver línea por línea qué hay ANTES de aplicar skiprows
lineas = clean_text.split('\r\n')
for i, linea in enumerate(lineas[:12]):
    print(f"Línea {i}: {linea[:100]}")

In [0]:
file_stream = io.StringIO(clean_text)
pdf = pd.read_csv(
    file_stream,
    skiprows=8,
    sep=",",
    header=0,
    engine='python',
    encoding_errors='ignore',
    on_bad_lines='skip',
    keep_default_na=True
)
print(pdf.columns.tolist())
print(pdf.head(5))

In [0]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Forma del DataFrame:", pdf.shape)
print("\nColumnas:", list(pdf.columns))
print("\nÍndice:", pdf.index.tolist()[:5])
print("\nPrimera fila completa como diccionario:")
print(pdf.iloc[0].to_dict())

In [0]:
file_stream = io.StringIO(clean_text)

# Leer solo la línea de encabezados para saber los nombres reales
lineas_texto = clean_text.split('\r\n')
header_line = lineas_texto[8]  # la línea de encabezados real
import csv
nombres_columnas = next(csv.reader([header_line]))
nombres_columnas.append("_columna_extra")  # nombre para el campo fantasma del final

file_stream = io.StringIO(clean_text)
pdf = pd.read_csv(
    file_stream,
    skiprows=9,              # ahora saltamos también la línea de encabezado (8) porque la ponemos manual
    sep=",",
    header=None,             # le decimos que NO hay encabezado en el archivo
    names=nombres_columnas,  # nosotros le damos los 11 nombres
    engine='python',
    encoding_errors='ignore',
    on_bad_lines='skip',
    keep_default_na=True,
    index_col=False
)

print(pdf.shape)
print(pdf.columns.tolist())
print(pdf.iloc[0].to_dict())

In [0]:
import requests

url_covid = "https://1drv.ms/x/c/9b3df75ac5d97177/IQA952j9q_4-RoJeMFhCG5CqAQ79T6BcFqHtAI5SfbpsnHg?e=zDrJgq"
download_url = url_covid.split("?")[0] + "?download=1"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"}

response = requests.get(download_url, headers=headers, allow_redirects=True)

print("URL final después de redirecciones:", response.url)
print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("Historial de redirecciones:")
for r in response.history:
    print(" ->", r.status_code, r.url)

In [0]:
print("Primeros 500 caracteres:")
print(response.content[:500])